# Thermal Equilibrium Test for Redfield Relaxation

## What is tested

`RedfieldRelaxationChannel` can enforce thermal detailed balance on its spectral density.
At thermal equilibrium the spectral density must obey the Kubo-Martin-Schwinger condition

$$J(-\omega) = J(\omega) \, \exp\left(-\frac{\hbar\omega}{k_B T}\right),$$

where $\omega > 0$ is a downward transition (emission) and $\omega < 0$ is an upward
transition (absorption). The notebook checks the three thermal balance modes:

- **skip**: the spectral density is not modified.
- **symmetric**: detailed balance is enforced on all transitions.
- **complement**: missing upward transitions are filled from the downward ones.

**Conventions used in this notebook**

- `omega_hz[i, j] = E_j - E_i` in Hz.
- `J[i, j]` is the spectral density of the transition $j \to i$.
- `W[a, c]` is the rate of the transition $c \to a$.

## List of tests

1. `test_boltzmann_factors`: the Boltzmann factors used by the tests are correct (unit diagonal, reciprocity, correct side of 1).
2. `test_skip_mode_unmodified`: skip mode returns the raw spectral density.
3. `test_skip_mode_no_thermal_bias`: skip mode with a constant density gives a symmetric rate matrix.
4. `test_symmetric_mode_detailed_balance_spectral`: symmetric mode satisfies detailed balance in the spectral density.
5. `test_symmetric_mode_detailed_balance_rates`: symmetric mode satisfies detailed balance in the rates.
6. `test_complement_mode_fills_upward_zeros`: complement mode fills zero upward entries from their downward partners.
7. `test_complement_mode_preserves_nonzero_upward`: complement mode keeps nonzero upward entries and adjusts the downward ones.
8. `test_symmetric_equilibrium_steady_state`: the Boltzmann populations are stationary in symmetric mode.
9. `test_complement_equilibrium_steady_state`: the Boltzmann populations are stationary in complement mode.
10. `test_skip_mode_breaks_equilibrium`: with an asymmetric density the Boltzmann populations are not stationary in skip mode.
11. `test_omega_conversion_to_rad_per_second`: the spectral density receives frequencies in rad/s.
12. `test_invalid_mode_raises`: invalid mode names and types raise errors.

## How to run

Run all cells. The last cell calls `run_tests()`. Each test prints one line when it
passes and raises `AssertionError` when it fails.

In [1]:
import math
import os
import sys
import typing as tp

import torch

# Makes MaRs importable when the notebook is run from the repository without installation
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

from mars.population import RedfieldRelaxationChannel

dtype = torch.float64

H_OVER_KB = 4.799243073e-11  # h / k_B in K / Hz
RELATIVE_TOLERANCE = 1e-6  # The thermal factors differ from 1 by less than 1 % at 77 K, so the tolerance must be tight

## Helper functions

### Model systems

In [2]:
def make_two_level(delta_hz: float) -> tuple[torch.Tensor, torch.Tensor]:
    """Return the energies [Hz] and a random Hermitian coupling operator of a two-level system."""
    energies = torch.tensor([0.0, delta_hz], dtype=dtype)
    torch.manual_seed(42)
    operator = torch.randn(2, 2, dtype=dtype)
    return energies, (operator + operator.T) / 2


def make_three_level() -> tuple[torch.Tensor, torch.Tensor]:
    """Return the energies [Hz] and a random Hermitian coupling operator of a three-level ladder."""
    energies = torch.tensor([0.0, 5e9, 8.5e9], dtype=dtype)
    torch.manual_seed(7)
    operator = torch.randn(3, 3, dtype=dtype)
    return energies, (operator + operator.T) / 2

### Spectral densities

In [3]:
def flat_spectral_density(omega_rad_s: torch.Tensor, temperature: tp.Optional[torch.Tensor]) -> torch.Tensor:
    """Return J(omega) = 1, which has no thermal bias."""
    return torch.ones_like(omega_rad_s)


def asymmetric_spectral_density(omega_rad_s: torch.Tensor, temperature: tp.Optional[torch.Tensor]) -> torch.Tensor:
    """Return J(omega) = 1 for omega > 0 and 0.5 otherwise, which deliberately breaks detailed balance."""
    return torch.where(omega_rad_s > 0, torch.ones_like(omega_rad_s), 0.5 * torch.ones_like(omega_rad_s))


def downward_only_spectral_density(omega_rad_s: torch.Tensor, temperature: tp.Optional[torch.Tensor]) -> torch.Tensor:
    """Return J(omega) = 1 for omega > 0 and 0 otherwise, so only emission is present."""
    return torch.where(omega_rad_s > 0, torch.ones_like(omega_rad_s), torch.zeros_like(omega_rad_s))

### Calculation helpers

In [4]:
def hz_to_kelvin(omega_hz: torch.Tensor) -> torch.Tensor:
    """Convert a frequency in Hz to an energy difference in K."""
    return omega_hz * H_OVER_KB


def build_channel(operator: torch.Tensor, spectral_density: tp.Callable, mode: str) -> RedfieldRelaxationChannel:
    """Create a secular Redfield channel in the eigenbasis with the given thermal balance mode."""
    channel = RedfieldRelaxationChannel(
        operator_components=[(operator, None)],
        spectral_density_func=spectral_density,
        thermal_balance_mode=mode,
    )
    channel.post_init(eigen_basis_flag=True, secular=True)
    return channel


def spectral_density_matrix(channel, energies: torch.Tensor, temperature: torch.Tensor) -> torch.Tensor:
    """Return the spectral density matrix J[i, j] of the channel."""
    return channel._get_spectral_density_matrix(energies, temperature)


def transition_frequencies(energies: torch.Tensor) -> torch.Tensor:
    """Return the matrix omega_hz[i, j] = E_j - E_i in Hz."""
    return energies.unsqueeze(-2) - energies.unsqueeze(-1)


def boltzmann_matrix(omega_hz: torch.Tensor, temperature: torch.Tensor) -> torch.Tensor:
    """Return the matrix of Boltzmann factors exp(-h omega_hz / k_B T)."""
    return torch.exp(-hz_to_kelvin(omega_hz) / temperature)


def boltzmann_distribution(energies: torch.Tensor, temperature: torch.Tensor) -> torch.Tensor:
    """Return the normalized Boltzmann populations."""
    exponents = -hz_to_kelvin(energies) / temperature
    populations = torch.exp(exponents - exponents.max())  # Shifted for numerical stability
    return populations / populations.sum()


def build_rate_matrix(rates: torch.Tensor) -> torch.Tensor:
    """Add the population drain -sum_a W[a, c] to the diagonal of a matrix of transition rates."""
    rate_matrix = rates.clone()
    index = torch.arange(rates.shape[-1])
    rate_matrix[..., index, index] = -rates.sum(dim=-2)
    return rate_matrix


def boltzmann_population_change(spectral_density: tp.Callable, mode: str) -> torch.Tensor:
    """Return dP/dt at the Boltzmann populations of the three-level system at 77 K."""
    energies, operator = make_three_level()
    temperature = torch.tensor(77.0, dtype=dtype)
    channel = build_channel(operator, spectral_density, mode)
    rates = channel.transition_rates(None, None, energies.unsqueeze(-2), temperature)
    return build_rate_matrix(rates) @ boltzmann_distribution(energies, temperature)

## Tests

### Boltzmann factors

In [5]:
def test_boltzmann_factors():
    """Check the unit diagonal, reciprocity and range of the Boltzmann factors used by the tests."""
    energies, _ = make_three_level()
    atol = 1e-10

    for temperature_value in [4.2, 77.0, 300.0]:
        temperature = torch.tensor(temperature_value, dtype=dtype)
        omega = transition_frequencies(energies)
        factors = boltzmann_matrix(omega, temperature)

        diagonal = torch.diagonal(factors)
        assert torch.allclose(diagonal, torch.ones_like(diagonal), atol=atol), (
            f"T={temperature_value}: the diagonal factors must be 1"
        )

        product = factors * factors.T
        assert torch.allclose(product, torch.ones_like(product), atol=atol), (
            f"T={temperature_value}: B[i,j] * B[j,i] must be 1"
        )

        assert torch.all(factors[omega > 0] <= 1.0 + atol), (
            f"T={temperature_value}: factors must not exceed 1 for omega > 0"
        )
        assert torch.all(factors[omega < 0] >= 1.0 - atol), (
            f"T={temperature_value}: factors must be at least 1 for omega < 0"
        )
    print("Boltzmann factors test passed (unit diagonal, reciprocity, correct range)")

### Skip mode

In [6]:
def test_skip_mode_unmodified():
    """Check that skip mode returns the raw spectral density."""
    energies, operator = make_three_level()
    temperature = torch.tensor(300.0, dtype=dtype)
    channel = build_channel(operator, flat_spectral_density, "skip")

    density = spectral_density_matrix(channel, energies, temperature)

    assert torch.allclose(density, torch.ones_like(density), atol=1e-8), "Skip mode must not change J"
    print("Skip mode test passed (the spectral density is not modified)")


def test_skip_mode_no_thermal_bias():
    """Check that skip mode with a constant density gives a symmetric rate matrix."""
    energies, operator = make_three_level()
    temperature = torch.tensor(300.0, dtype=dtype)
    channel = build_channel(operator, flat_spectral_density, "skip")

    rates = channel.transition_rates(None, None, energies.unsqueeze(-2), temperature)

    assert torch.allclose(rates, rates.transpose(-2, -1), atol=1e-8), (
        "Skip mode with a constant J must give symmetric rates"
    )
    print("Skip mode bias test passed (a constant density gives symmetric rates)")

### Symmetric mode

In [7]:
def test_symmetric_mode_detailed_balance_spectral():
    """Check the Kubo-Martin-Schwinger condition for every off-diagonal spectral density entry."""
    energies, operator = make_three_level()
    dim = energies.shape[-1]

    for temperature_value in [4.2, 77.0, 300.0]:
        temperature = torch.tensor(temperature_value, dtype=dtype)
        # The asymmetric density makes sure that the mode has something to correct
        channel = build_channel(operator, asymmetric_spectral_density, "symmetric")
        density = spectral_density_matrix(channel, energies, temperature)
        expected = boltzmann_matrix(transition_frequencies(energies), temperature).transpose(-2, -1)

        for i in range(dim):
            for j in range(dim):
                if i == j or density[j, i].item() < 1e-30:
                    continue
                ratio = (density[i, j] / density[j, i]).item()
                assert abs(ratio - expected[i, j].item()) < RELATIVE_TOLERANCE * abs(expected[i, j].item()), (
                    f"T={temperature_value}: detailed balance violated at [{i},{j}]: "
                    f"ratio {ratio:.6f}, expected {expected[i, j].item():.6f}"
                )
    print("Symmetric mode spectral test passed (J satisfies detailed balance at 4.2, 77 and 300 K)")


def test_symmetric_mode_detailed_balance_rates():
    """Check that the transition rates of symmetric mode satisfy detailed balance."""
    energies, operator = make_three_level()
    temperature = torch.tensor(77.0, dtype=dtype)
    channel = build_channel(operator, asymmetric_spectral_density, "symmetric")
    rates = channel.transition_rates(None, None, energies.unsqueeze(-2), temperature)
    expected = boltzmann_matrix(transition_frequencies(energies), temperature).transpose(-2, -1)

    dim = energies.shape[-1]
    checked_pairs = 0
    for a in range(dim):
        for c in range(dim):
            if a == c or rates[0, a, c].item() < 1e-30 or rates[0, c, a].item() < 1e-30:
                continue
            ratio = (rates[0, a, c] / rates[0, c, a]).item()
            assert abs(ratio - expected[a, c].item()) < RELATIVE_TOLERANCE * abs(expected[a, c].item()), (
                f"Detailed balance violated for W[{a},{c}] / W[{c},{a}]: "
                f"ratio {ratio:.6f}, expected {expected[a, c].item():.6f}"
            )
            checked_pairs += 1

    assert checked_pairs > 0, "No transition pairs were checked"
    print(f"Symmetric mode rates test passed (detailed balance holds for {checked_pairs} transition pairs)")

### Complement mode

In [8]:
def test_complement_mode_fills_upward_zeros():
    """Check that zero upward entries are filled from the downward ones and downward entries are kept."""
    energies, operator = make_three_level()
    temperature = torch.tensor(77.0, dtype=dtype)
    channel = build_channel(operator, downward_only_spectral_density, "complement")
    density = spectral_density_matrix(channel, energies, temperature)
    omega = transition_frequencies(energies)
    dim = energies.shape[-1]

    for i in range(dim):
        for j in range(dim):
            if i == j:
                continue
            value = density[i, j].item()
            if omega[i, j].item() > 0:
                assert value > 0, f"Downward J[{i},{j}] must be kept"
            elif omega[i, j].item() < 0:
                assert value > 0, f"Upward J[{i},{j}] must be filled"
                partner = density[j, i].item()
                expected = partner * math.exp(-hz_to_kelvin(omega[j, i]).item() / temperature.item())
                assert abs(value - expected) < RELATIVE_TOLERANCE * expected, (
                    f"J[{i},{j}] = {value:.6e}, expected {expected:.6e}"
                )
    print("Complement mode filling test passed (zero upward entries follow detailed balance)")


def test_complement_mode_preserves_nonzero_upward():
    """Check that nonzero upward entries are kept and downward entries satisfy detailed balance."""
    energies, operator = make_three_level()
    temperature = torch.tensor(77.0, dtype=dtype)
    channel = build_channel(operator, flat_spectral_density, "complement")
    density = spectral_density_matrix(channel, energies, temperature)
    omega = transition_frequencies(energies)
    dim = energies.shape[-1]
    one = torch.tensor(1.0, dtype=dtype)

    for i in range(dim):
        for j in range(dim):
            if i == j:
                assert torch.isclose(density[i, j], one, atol=1e-8), f"Diagonal J[{i},{i}] must stay 1"
            elif omega[i, j].item() < 0:
                assert torch.isclose(density[i, j], one, atol=1e-8), (
                    f"Upward J[{i},{j}] must stay 1, got {density[i, j].item()}"
                )
            else:
                # Downward = upward partner * exp(+dE / k_B T), so the downward rate is the larger one
                expected = density[j, i].item() * math.exp(hz_to_kelvin(omega[i, j]).item() / temperature.item())
                assert abs(density[i, j].item() - expected) < RELATIVE_TOLERANCE * expected, (
                    f"Downward J[{i},{j}] = {density[i, j].item():.6e}, expected {expected:.6e}"
                )
    print("Complement mode preservation test passed (upward entries are kept, downward entries are adjusted)")

### Equilibrium populations

In [9]:
def test_symmetric_equilibrium_steady_state():
    """Check that the Boltzmann populations are stationary in symmetric mode."""
    change = boltzmann_population_change(asymmetric_spectral_density, "symmetric")
    assert torch.allclose(change, torch.zeros_like(change), atol=1e-5), (
        f"Boltzmann populations are not stationary. Max |dP/dt| = {change.abs().max().item():.2e}"
    )
    print("Symmetric mode equilibrium test passed (Boltzmann populations are stationary)")


def test_complement_equilibrium_steady_state():
    """Check that the Boltzmann populations are stationary in complement mode."""
    change = boltzmann_population_change(downward_only_spectral_density, "complement")
    assert torch.allclose(change, torch.zeros_like(change), atol=1e-5), (
        f"Boltzmann populations are not stationary. Max |dP/dt| = {change.abs().max().item():.2e}"
    )
    print("Complement mode equilibrium test passed (Boltzmann populations are stationary)")


def test_skip_mode_breaks_equilibrium():
    """Check that skip mode with an asymmetric density does not make the Boltzmann populations stationary."""
    change = boltzmann_population_change(asymmetric_spectral_density, "skip")
    assert not torch.allclose(change, torch.zeros_like(change), atol=1e-5), (
        "Skip mode with an asymmetric J must not give a Boltzmann steady state"
    )
    print("Skip mode equilibrium test passed (Boltzmann populations are not stationary, as expected)")

### Input handling

In [10]:
def test_omega_conversion_to_rad_per_second():
    """Check that the spectral density function receives frequencies of magnitude 2 pi * omega_hz."""
    received = []

    def recording_spectral_density(omega_rad_s: torch.Tensor, temperature: tp.Optional[torch.Tensor]) -> torch.Tensor:
        """Store the received frequencies and return J(omega) = 1."""
        received.append(omega_rad_s.clone())
        return torch.ones_like(omega_rad_s)

    delta_hz = 1e10
    energies, operator = make_two_level(delta_hz=delta_hz)
    temperature = torch.tensor(300.0, dtype=dtype)
    channel = build_channel(operator, recording_spectral_density, "skip")
    spectral_density_matrix(channel, energies, temperature)

    assert len(received) > 0, "The spectral density function was never called"
    expected = 2 * math.pi * delta_hz
    actual = received[0][0, 1].abs().item()
    relative_error = abs(actual - expected) / expected
    assert relative_error < 1e-6, f"Received omega = {actual:.4e} rad/s, expected {expected:.4e} rad/s"
    print("Frequency conversion test passed (the spectral density receives rad/s)")


def test_invalid_mode_raises():
    """Check that an unknown mode name raises ValueError and a wrong mode type raises TypeError."""
    _, operator = make_two_level(delta_hz=1e9)

    for invalid_mode, expected_error in [("nonsense", ValueError), (42, TypeError)]:
        try:
            RedfieldRelaxationChannel(
                operator_components=[(operator, None)],
                spectral_density_func=flat_spectral_density,
                thermal_balance_mode=invalid_mode,
            )
        except expected_error:
            continue
        raise AssertionError(f"Mode {invalid_mode!r} must raise {expected_error.__name__}")
    print("Invalid mode test passed (wrong names and types raise errors)")

## Run all tests

In [11]:
def run_tests():
    """Run every test of this notebook."""
    tests = [
        test_boltzmann_factors,
        test_skip_mode_unmodified,
        test_skip_mode_no_thermal_bias,
        test_symmetric_mode_detailed_balance_spectral,
        test_symmetric_mode_detailed_balance_rates,
        test_complement_mode_fills_upward_zeros,
        test_complement_mode_preserves_nonzero_upward,
        test_symmetric_equilibrium_steady_state,
        test_complement_equilibrium_steady_state,
        test_skip_mode_breaks_equilibrium,
        test_omega_conversion_to_rad_per_second,
        test_invalid_mode_raises,
    ]
    for test in tests:
        test()
    print(f"\nAll {len(tests)} Redfield thermal equilibrium tests passed.")


run_tests()

Boltzmann factors test passed (unit diagonal, reciprocity, correct range)
Skip mode test passed (the spectral density is not modified)
Skip mode bias test passed (a constant density gives symmetric rates)
Symmetric mode spectral test passed (J satisfies detailed balance at 4.2, 77 and 300 K)
Symmetric mode rates test passed (detailed balance holds for 6 transition pairs)
Complement mode filling test passed (zero upward entries follow detailed balance)
Complement mode preservation test passed (upward entries are kept, downward entries are adjusted)
Symmetric mode equilibrium test passed (Boltzmann populations are stationary)
Complement mode equilibrium test passed (Boltzmann populations are stationary)
Skip mode equilibrium test passed (Boltzmann populations are not stationary, as expected)
Frequency conversion test passed (the spectral density receives rad/s)
Invalid mode test passed (wrong names and types raise errors)

All 12 Redfield thermal equilibrium tests passed.
